# DuckDB SQL Console

Interactive read-only console for the per-experiment DuckDB files produced by `scripts/run_analysis.py` and `scripts/run_experiment_report.py`.

Each `<experiment_id>.db` lives under the project's `results/` directory and follows the schema defined in `src/analysis/analysis.py::LDPCAnalysis.create_database_schema` (three tables: `experiments`, `episodes`, `steps`).

Workflow: enter a database name, type a query, click *Validate* (uses `EXPLAIN`), then *Execute*. Results are paginated.

In [ ]:
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / 'pyproject.toml').exists() and (_p / 'src').exists():
        sys.path.insert(0, str(_p))
        break

import duckdb
import ipywidgets as widgets
from IPython.display import clear_output, display

from src.utils.path_utils import resolve_results_path

PAGE_SIZE = 50
DEFAULT_RESULTS_DIR = str(resolve_results_path())

In [ ]:
def database_path(folder: str, name: str) -> Path:
    if not name.strip():
        raise ValueError('Database name is empty')
    path = Path(folder) / (name if name.endswith('.db') else name + '.db')
    if not path.exists():
        raise FileNotFoundError(f'Database not found: {path}')
    return path


def validate(query: str, folder: str, name: str) -> tuple[bool, str]:
    if not query.strip():
        return False, 'Query is empty'
    try:
        path = database_path(folder, name)
        with duckdb.connect(str(path), read_only=True) as con:
            con.execute(f'EXPLAIN {query}')
        return True, 'Query syntax is valid'
    except (ValueError, FileNotFoundError) as e:
        return False, str(e)
    except Exception as e:
        return False, f'Validation error: {e}'


def execute(query: str, folder: str, name: str):
    path = database_path(folder, name)
    with duckdb.connect(str(path), read_only=True) as con:
        return con.execute(query).df()


class Paginator:
    def __init__(self, df, page_size=PAGE_SIZE):
        self.df = df
        self.page_size = page_size
        self.page = 0
        self.total_pages = max(1, (len(df) + page_size - 1) // page_size) if len(df) else 0

    def current(self):
        start = self.page * self.page_size
        return self.df.iloc[start:start + self.page_size]

    def info(self) -> str:
        if not self.total_pages:
            return 'No rows'
        start = self.page * self.page_size + 1
        end = min(start + self.page_size - 1, len(self.df))
        return f'Rows {start}-{end} of {len(self.df)} | Page {self.page + 1}/{self.total_pages}'

In [ ]:
folder_input = widgets.Text(value=DEFAULT_RESULTS_DIR, description='Folder:', layout=widgets.Layout(width='600px'))
database_input = widgets.Text(value='', placeholder='<experiment_id>.db', description='Database:', layout=widgets.Layout(width='600px'))
query_input = widgets.Textarea(value='', placeholder='SELECT * FROM episodes LIMIT 10;', description='SQL:', layout=widgets.Layout(width='800px', height='160px'))

validate_btn = widgets.Button(description='Validate', button_style='info')
execute_btn = widgets.Button(description='Execute', button_style='success')
prev_btn = widgets.Button(description='Previous', disabled=True)
next_btn = widgets.Button(description='Next', disabled=True)

status = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='4px', width='800px'))
results = widgets.Output(layout=widgets.Layout(width='100%', overflow='auto'))

paginator: Paginator | None = None


def render_page():
    with results:
        clear_output(wait=True)
        if paginator is None or paginator.total_pages == 0:
            print('No results to display')
        else:
            display(paginator.current())
            print('\n' + paginator.info())
    prev_btn.disabled = paginator is None or paginator.page == 0
    next_btn.disabled = paginator is None or paginator.page >= paginator.total_pages - 1


def on_validate(_):
    ok, msg = validate(query_input.value, folder_input.value, database_input.value)
    with status:
        clear_output(wait=True)
        print(('OK: ' if ok else 'ERROR: ') + msg)


def on_execute(_):
    global paginator
    ok, msg = validate(query_input.value, folder_input.value, database_input.value)
    if not ok:
        with status:
            clear_output(wait=True)
            print('ERROR: ' + msg)
        return
    try:
        df = execute(query_input.value, folder_input.value, database_input.value)
        paginator = Paginator(df)
        with status:
            clear_output(wait=True)
            print(f'OK: returned {len(df)} row(s)')
        render_page()
    except Exception as e:
        paginator = None
        with status:
            clear_output(wait=True)
            print(f'ERROR: {e}')
        render_page()


def on_prev(_):
    if paginator and paginator.page > 0:
        paginator.page -= 1
        render_page()


def on_next(_):
    if paginator and paginator.page < paginator.total_pages - 1:
        paginator.page += 1
        render_page()


validate_btn.on_click(on_validate)
execute_btn.on_click(on_execute)
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)

ui = widgets.VBox([
    widgets.HTML('<h3>Connection</h3>'), folder_input, database_input,
    widgets.HTML('<h3>Query</h3>'), query_input,
    widgets.HBox([validate_btn, execute_btn]),
    widgets.HTML('<h4>Status</h4>'), status,
    widgets.HTML('<h3>Results</h3>'), widgets.HBox([prev_btn, next_btn]), results,
])
display(ui)

## Schema reference

Defined in `src/analysis/analysis.py::create_database_schema`.

- `experiments(experiment_id, experiment_name, configuration JSON, created_at)`
- `episodes(global_episode_id, experiment_id, episode_type, original_episode_id, agent_type, phase, original_codeword, noisy_codeword, error_pattern, error_model, num_errors, error_parameters, terminal_reason, episode_length, success)`
- `steps(step_id, global_episode_id, step_number, action, reward, done, success, codeword_weight, syndrome_weight, flip_weight, penalty_sum, step_count, next_codeword_weight, next_syndrome_weight, next_flip_weight, next_penalty_sum, ...)` — training updates (loss, epsilon, q-value stats) are nested under each step row as JSON, not flat columns.

## Example queries

List tables and columns:
```sql
SHOW TABLES;
DESCRIBE episodes;
```

Top-level success rate per agent (matches the `overall_performance_summary` analysis):
```sql
SELECT agent_type,
       CAST(SUM(CASE WHEN terminal_reason = 'SUCCESS_STOP' THEN 1 ELSE 0 END) AS REAL) / COUNT(*) AS success_rate,
       AVG(episode_length) AS avg_steps
FROM   episodes
WHERE  episode_type = 'eval'
GROUP  BY agent_type
ORDER  BY success_rate DESC;
```

Per-phase training summary:
```sql
SELECT phase,
       COUNT(*)                                                  AS episodes,
       SUM(CASE WHEN success THEN 1 ELSE 0 END)                  AS successful,
       ROUND(100.0 * SUM(CASE WHEN success THEN 1 ELSE 0 END) / COUNT(*), 2) AS success_pct,
       ROUND(AVG(episode_length), 2)                             AS avg_length
FROM   episodes
WHERE  episode_type = 'train'
GROUP  BY phase
ORDER  BY phase;
```

Sample of step-level data (compressed state weights):
```sql
SELECT step_id, global_episode_id, step_number, action, reward,
       syndrome_weight, flip_weight, next_syndrome_weight, next_flip_weight
FROM   steps
ORDER  BY step_id DESC
LIMIT  20;
```

Experiment configuration (JSON):
```sql
SELECT experiment_id, experiment_name,
       json_extract_string(configuration, '$.code_id') AS code_id,
       json_extract_string(configuration, '$.training.network.type') AS network
FROM   experiments;
```